In [58]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [59]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from numba import njit
from simulators import ContextManager, NestedModelFamily
from simulators.benchmarks import CollapsingBoundDDM, DDMModel

In [60]:
params = ["v_intercept", "v_slope", "s_v", "a_intercept", "a_slope", "decay", "tau", "s_tau"]

### Priors

In [61]:
@njit
def sample_ddm_prior():
    v_intercept = np.random.gamma(3.0, 0.8)
    v_slope     = np.random.normal(0.0, 3.0)
    s_v         = np.random.gamma(1.0, 0.2)
    a_intercept = np.random.gamma(10.0, 0.3)
    a_slope     = np.random.normal(0.0, 1.0)
    decay       = np.random.gamma(1.0, 0.4)
    tau         = np.random.gamma(3.0, 0.2)
    s_tau       = np.random.uniform(0.0, tau * 2.0)
    return np.array([v_intercept, v_slope, s_v, a_intercept, a_slope, decay, tau, s_tau], dtype=np.float32)

In [62]:
# Generate design matrix
num_samples = 100
design_mat = np.random.normal(0.0, 1.0, size=(num_samples, 2)).astype(np.float32)
x_v = design_mat[:, 0]
x_a = design_mat[:, 1]
context = ContextManager(parameter_names=params)

In [63]:
family = NestedModelFamily(
    name="ddm",
    model=DDMModel,
    context_manager=context,
    prior_fun=sample_ddm_prior,
    num_samples=num_samples
)

In [70]:
# Run multiple configs in one go
configs = [
    {"s_v", "s_tau"},                # fix variability terms
    {"s_v"},
    {"s_tau"},# fix only s_v
    set(),                           # all free
]

In [73]:
results = family.sample(configs, context={"x_v": x_v, "x_a": x_a})

In [74]:
results

[{'variant_name': 'ddm|cfg1',
  'fixed_parameters': ['s_tau', 's_v'],
  'mask': {'v_intercept': 1.0,
   'v_slope': 1.0,
   's_v': 0.0,
   'a_intercept': 1.0,
   'a_slope': 1.0,
   'decay': 1.0,
   'tau': 1.0,
   's_tau': 0.0},
  'prior_draw': array([ 1.9243768 , -3.6543975 ,  0.28203502,  2.7502675 , -1.3923181 ,
          0.0713124 ,  0.26350874,  0.10680419], dtype=float32),
  'full_params': {'v_intercept': 1.9243768,
   'v_slope': -3.6543975,
   's_v': 0.0,
   'a_intercept': 2.7502675,
   'a_slope': -1.3923181,
   'decay': 0.0713124,
   'tau': 0.26350874,
   's_tau': 0.0},
  'sim_data': array([[0.46150875, 1.        ],
         [2.205509  , 1.        ],
         [0.86950874, 1.        ],
         [0.34950873, 1.        ],
         [0.69350874, 1.        ],
         [0.42950875, 1.        ],
         [0.5975087 , 1.        ],
         [0.39950874, 1.        ],
         [8.849509  , 0.        ],
         [1.1435088 , 0.        ],
         [0.47450873, 1.        ],
         [0.57350874